# 04. 검증(Validate) & 지표 해석

학습이 잘 됐는지 **숫자로 판단**하는 법을 배웁니다.

배우는 것
1. Precision / Recall / mAP 가 뭔지
2. IoU 와 mAP50 vs mAP50-95
3. `model.val()` 결과 객체 뜯어보기
4. Confusion matrix / PR curve 읽기
5. 성능이 나쁠 때 어디를 볼까

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import torch
from ultralytics import YOLO

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
model = YOLO("yolo26n.pt")
print("device:", DEVICE)

## 1. 기본 개념

탐지 결과는 정답과 비교해 4가지로 분류됩니다:
- **TP** (True Positive): 맞게 탐지
- **FP** (False Positive): 없는 걸 탐지 (오탐)
- **FN** (False Negative): 있는 걸 놓침 (미탐)

여기서:

| 지표 | 식 | 의미 |
|---|---|---|
| **Precision** | TP / (TP+FP) | 탐지한 것 중 진짜 비율 ("오탐 적나?") |
| **Recall** | TP / (TP+FN) | 실제 객체 중 잡아낸 비율 ("놓치지 않나?") |

→ 둘은 보통 **트레이드오프**. conf를 낮추면 recall↑ precision↓.

## 2. IoU 와 mAP

**IoU (Intersection over Union)**: 예측 박스와 정답 박스의 겹침 정도 (0~1).

```
IoU = 겹치는 넓이 / 합친 넓이
```

IoU가 특정 임계값(예: 0.5) 이상이면 "맞게 탐지(TP)"로 인정.

**mAP (mean Average Precision)**: PR 곡선 아래 면적(AP)을 클래스마다 구해 평균.

| 지표 | 의미 |
|---|---|
| **mAP50** | IoU 0.5 기준. 관대함 (박스가 대충 맞아도 인정) |
| **mAP50-95** | IoU 0.5~0.95를 0.05 간격으로 평균. 엄격함 (박스 정밀도까지 봄) |

→ 논문·벤치마크의 대표 지표는 보통 **mAP50-95**. 높을수록 좋음.

## 3. model.val() 결과 뜯어보기

In [ ]:
metrics = model.val(data="coco8.yaml", device=DEVICE)

print("mAP50-95 :", round(metrics.box.map, 4))
print("mAP50    :", round(metrics.box.map50, 4))
print("mAP75    :", round(metrics.box.map75, 4))
print("평균 P    :", round(metrics.box.mp, 4))
print("평균 R    :", round(metrics.box.mr, 4))
print("결과 폴더 :", metrics.save_dir)

In [ ]:
# 클래스별 mAP50-95 (어떤 클래스가 약한지 확인)
import pandas as pd

names = model.names
rows = [{"class": names[c], "mAP50-95": round(float(m), 3)}
        for c, m in zip(metrics.box.ap_class_index, metrics.box.maps[metrics.box.ap_class_index])]
pd.DataFrame(rows).sort_values("mAP50-95")

## 4. 시각화 결과물 보기

`val()`은 폴더에 그래프를 저장합니다. 노트북에 띄워봅시다.

- **confusion_matrix.png**: 어떤 클래스를 어떤 클래스로 헷갈리는지
- **BoxPR_curve.png**: precision-recall 곡선 (아래 면적 = AP)
- **BoxF1_curve.png**: conf에 따른 F1 → 최적 conf 임계값 힌트

In [ ]:
from pathlib import Path
from PIL import Image

d = Path(metrics.save_dir)
print("생성된 그래프:")
for p in sorted(d.glob("*.png")):
    print("  ", p.name)

# 하나 띄워보기 (있으면)
pr = d / "BoxPR_curve.png"
Image.open(pr) if pr.exists() else print("PR curve 없음")

## 5. 성능이 나쁠 때 진단 가이드

| 증상 | 가능한 원인 | 시도 |
|---|---|---|
| Precision 낮음 (오탐 많음) | conf 너무 낮음, 배경 오탐 | conf 올리기, 배경 이미지 추가 |
| Recall 낮음 (많이 놓침) | 데이터 부족, 작은 객체 | imgsz 키우기, 데이터·증강 추가 |
| 특정 클래스만 낮음 | 그 클래스 데이터 적음/라벨 오류 | 데이터 보강, 라벨 재검수 |
| train mAP 높은데 val 낮음 | 과적합 | 데이터 늘리기, 증강↑, epochs↓ |
| 둘 다 낮음 | 학습 부족/lr 문제 | epochs↑, lr 조정 |

> **가장 흔한 원인은 데이터**입니다. 하이퍼파라미터보다 데이터 양·품질·라벨 정확도를 먼저 의심하세요.

**다음 노트북 →** `05_export_deploy` : 모델 내보내기와 배포

In [ ]:
# 연습 공간: conf 를 바꿔가며 P/R 이 어떻게 변하는지 관찰해보세요
